In [0]:
# -----------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 4.4_ranking_fornecedores_suspeitos
# LOCAL: 3_gold/src/4_gastos_ceap/
# OBJETIVO: Gerar ranking de fornecedores mais pagos a partir da tabela gerada no script 3_gold/4_gastos_ceap/4.2_fato_ceap_despesas 
# ENTREGA: Ranking de fornecedores mais pagos com flags de CNPJ suspeito (consulta Receita)
# -----------------------------------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, count, sum, desc, length

# Agrupar por fornecedor para identificar volume de pagamentos
df_fornecedores = spark.table("gold_fato_ceap_despesas") \
    .groupBy("cnpj_fornecedor", "nome_fornecedor") \
    .agg(
        count("id_despesa").alias("qtd_notas"),
        sum("valor_gasto").alias("total_recebido")
    )

# Criar flags de suspeita (CNPJ incompleto ou volume muito alto de notas)
df_ranking = df_fornecedores.withColumn(
    "flag_cnpj_invalido", 
    (length(col("cnpj_fornecedor")) < 14) 
).orderBy(desc("total_recebido"))

# Salvar na Gold
df_ranking.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_ranking_fornecedores_ceap")

display(df_ranking)